# 🗄️ Week 6 — SQL Group Exercise

## Team Database Challenge

**Welcome to the SQL Group Session!** Work with your team to complete these SQL challenges.

---

### 🎯 Session Goals

- ✅ Practice SQL queries as a team
- ✅ Build a complete data analysis workflow
- ✅ Combine SQL with Pandas for insights
- ✅ Create visualizations from database queries

### 👥 Team Roles

Assign roles within your group:

| Role | Responsibility |
|------|---------------|
| **Driver** | Types the code |
| **Navigator** | Guides query logic |
| **Researcher** | Looks up SQL syntax |
| **Presenter** | Explains results to class |

**⏱️ Time:** 45 minutes total

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🚀 SETUP: Initialize the database (works in Colab & local Jupyter)
# ═══════════════════════════════════════════════════════════════════════════════

import sqlite3
import os
from pathlib import Path

# Check if we're in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("☁️  Running in Google Colab - setting up environment...")
    
    # Clone the repo if not already present
    if not os.path.exists('Python-Essentials-Code-The-Dream'):
        !git clone https://github.com/StrayDogSyn/Python-Essentials-Code-The-Dream.git
        print("✅ Repository cloned!")
    
    # Change to week8 directory
    os.chdir('Python-Essentials-Code-The-Dream/notebook-sessions/week8')
    print(f"📁 Working directory: {os.getcwd()}")
    
except ImportError:
    IN_COLAB = False
    print("💻 Running locally")

# Now set up the database
DB_PATH = "test.db"
SQL_PATH = "test.sql"

# Check if SQL file exists
if os.path.exists(SQL_PATH):
    # Remove existing database
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print("🗑️  Removed existing database")
    
    # Read and execute SQL
    with open(SQL_PATH, 'r', encoding='utf-8') as f:
        sql_script = f.read()
    
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript(sql_script)
        
        # Verify tables
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
        tables = cursor.fetchall()
        
    print("═" * 60)
    print("✅ Database created successfully!")
    print(f"📊 Tables: {', '.join([t[0] for t in tables])}")
    print("═" * 60)
else:
    print("⚠️  SQL file not found. Make sure you're in the week8 directory.")
    print(f"   Looking for: {os.path.abspath(SQL_PATH)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📦 IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = "test.db"

# Helper function for running queries
def run_query(query: str) -> pd.DataFrame:
    """Execute a SQL query and return results as DataFrame."""
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql(query, conn)

print("✅ Setup complete! Use run_query() to execute SQL.")

---

## 🏆 Challenge 1: Music Discovery Report (10 min)

**Scenario:** Your music streaming company wants a report on artist diversity.

### Tasks:

1. Find all artists grouped by country
2. Calculate the number of albums per country
3. Find the earliest and latest album per country

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 1.1: Artists by Country
# ═══════════════════════════════════════════════════════════════════════════════
# Write a query to list all artists with their country, sorted by country

query = """
    -- TODO: Your SQL here
    SELECT name, country, genre, formed_year
    FROM artists
    ORDER BY country, name
"""

df = run_query(query)
print("🌍 Artists by Country:")
df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 1.2: Albums per Country (requires JOIN + GROUP BY)
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Join artists and albums, group by country
    SELECT 
        artists.country,
        COUNT(albums.album_id) AS total_albums,
        COUNT(DISTINCT artists.artist_id) AS num_artists,
        MIN(albums.release_year) AS earliest_album,
        MAX(albums.release_year) AS latest_album
    FROM artists
    LEFT JOIN albums ON artists.artist_id = albums.artist_id
    GROUP BY artists.country
    ORDER BY total_albums DESC
"""

df_country = run_query(query)
print("📊 Album Statistics by Country:")
df_country

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 Visualize: Albums by Country
# ═══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#00f0ff', '#ff00ff', '#00ff41', '#ffff00']

bars = ax.bar(df_country['country'], df_country['total_albums'], color=colors[:len(df_country)])
ax.set_xlabel('Country')
ax.set_ylabel('Number of Albums')
ax.set_title('🌍 Albums by Artist Country')

# Style
ax.set_facecolor('#1a1f3a')
fig.patch.set_facecolor('#0a0e27')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('#00f0ff')
for spine in ax.spines.values():
    spine.set_color('#00f0ff')

plt.tight_layout()
plt.show()

---

## 🏆 Challenge 2: Sales Performance Analysis (15 min)

**Scenario:** The marketing team needs sales insights to plan their next campaign.

### Tasks:

1. Calculate total revenue by format (CD, Vinyl, Digital, Streaming)
2. Find the top 5 best-selling albums with artist names
3. Analyze sales trends by month

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 2.1: Revenue by Format
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Group sales by format, calculate totals
    SELECT 
        format,
        COUNT(*) AS num_transactions,
        SUM(quantity) AS units_sold,
        ROUND(SUM(quantity * unit_price), 2) AS total_revenue,
        ROUND(AVG(unit_price), 2) AS avg_price
    FROM album_sales
    GROUP BY format
    ORDER BY total_revenue DESC
"""

df_format = run_query(query)
print("💿 Sales by Format:")
df_format

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 2.2: Top 5 Best-Selling Albums (with artist names)
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Join album_sales → albums → artists, aggregate by album
    SELECT 
        albums.title AS album,
        artists.name AS artist,
        albums.genre,
        SUM(album_sales.quantity) AS units_sold,
        ROUND(SUM(album_sales.quantity * album_sales.unit_price), 2) AS revenue
    FROM album_sales
    INNER JOIN albums ON album_sales.album_id = albums.album_id
    INNER JOIN artists ON albums.artist_id = artists.artist_id
    GROUP BY albums.album_id
    ORDER BY revenue DESC
    LIMIT 5
"""

df_top = run_query(query)
print("🏆 Top 5 Best-Selling Albums:")
df_top

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 2.3: Monthly Sales Trend
# ═══════════════════════════════════════════════════════════════════════════════
# Hint: Use substr(sale_date, 1, 7) to get YYYY-MM

query = """
    -- TODO: Extract month from sale_date, aggregate
    SELECT 
        substr(sale_date, 1, 7) AS month,
        COUNT(*) AS transactions,
        SUM(quantity) AS units,
        ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM album_sales
    GROUP BY month
    ORDER BY month
"""

df_monthly = run_query(query)
print("📅 Monthly Sales:")
df_monthly

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 Visualize: Sales Dashboard
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e27')

# Left: Revenue by Format (pie chart)
ax1 = axes[0]
colors_pie = ['#00f0ff', '#ff00ff', '#00ff41', '#ffff00']
ax1.pie(df_format['total_revenue'], labels=df_format['format'], autopct='%1.1f%%',
        colors=colors_pie, startangle=90)
ax1.set_title('💿 Revenue by Format', color='#00f0ff', fontsize=14)

# Right: Monthly Trend (line chart)
ax2 = axes[1]
ax2.plot(df_monthly['month'], df_monthly['revenue'], marker='o', 
         color='#00f0ff', linewidth=2, markersize=8)
ax2.fill_between(df_monthly['month'], df_monthly['revenue'], alpha=0.3, color='#00f0ff')
ax2.set_xlabel('Month', color='white')
ax2.set_ylabel('Revenue ($)', color='white')
ax2.set_title('📈 Monthly Revenue Trend', color='#00f0ff', fontsize=14)
ax2.set_facecolor('#1a1f3a')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_color('#00f0ff')

plt.tight_layout()
plt.show()

print(f"\n💰 Total Revenue: ${df_format['total_revenue'].sum():.2f}")

---

## 🏆 Challenge 3: Playlist Insights (10 min)

**Scenario:** Product team wants to understand playlist engagement patterns.

### Tasks:

1. Find how many tracks are in each playlist
2. Identify which artists appear most in playlists
3. Calculate total playlist duration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 3.1: Tracks per Playlist
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Join playlists with playlist_tracks, count tracks
    SELECT 
        playlists.name AS playlist,
        playlists.description,
        COUNT(playlist_tracks.track_id) AS num_tracks
    FROM playlists
    LEFT JOIN playlist_tracks ON playlists.playlist_id = playlist_tracks.playlist_id
    GROUP BY playlists.playlist_id
    ORDER BY num_tracks DESC
"""

df_playlists = run_query(query)
print("🎵 Playlist Summary:")
df_playlists

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 3.2: Most Popular Artists in Playlists
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Complex join to find artist popularity across playlists
    SELECT 
        artists.name AS artist,
        COUNT(playlist_tracks.track_id) AS playlist_appearances,
        COUNT(DISTINCT playlist_tracks.playlist_id) AS unique_playlists
    FROM playlist_tracks
    INNER JOIN tracks ON playlist_tracks.track_id = tracks.track_id
    INNER JOIN albums ON tracks.album_id = albums.album_id
    INNER JOIN artists ON albums.artist_id = artists.artist_id
    GROUP BY artists.artist_id
    ORDER BY playlist_appearances DESC
"""

df_artist_pop = run_query(query)
print("⭐ Most Featured Artists in Playlists:")
df_artist_pop

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 Task 3.3: Playlist Duration (Advanced)
# ═══════════════════════════════════════════════════════════════════════════════

query = """
    -- TODO: Calculate total duration of each playlist in minutes
    SELECT 
        playlists.name AS playlist,
        COUNT(tracks.track_id) AS num_tracks,
        ROUND(SUM(tracks.duration_seconds) / 60.0, 1) AS duration_minutes
    FROM playlists
    LEFT JOIN playlist_tracks ON playlists.playlist_id = playlist_tracks.playlist_id
    LEFT JOIN tracks ON playlist_tracks.track_id = tracks.track_id
    GROUP BY playlists.playlist_id
    ORDER BY duration_minutes DESC
"""

df_duration = run_query(query)
print("⏱️ Playlist Durations:")
df_duration

---

## 🏆 Bonus Challenge: Create Your Own Analysis (10 min)

**Challenge:** Come up with your own business question and answer it with SQL!

Some ideas:
- Which genre generates the most revenue per album?
- What's the average album age (years since release) in sales?
- Which region prefers which format?
- Create a "New Releases" playlist recommendation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 YOUR TURN: Write your own analysis
# ═══════════════════════════════════════════════════════════════════════════════

# Business question: _______________________________

query = """
    -- Your SQL here!
    
"""

# Uncomment to run:
# df_custom = run_query(query)
# df_custom

---

## 📝 Group Reflection

Before presenting, discuss with your team:

1. **What was the most challenging query?** Why?
2. **Which JOIN type did you use most?** (INNER, LEFT, etc.)
3. **How does SQL compare to Pandas** for this type of analysis?
4. **What business insight surprised you?**

---

## 🎉 Great Teamwork!

You've practiced:
- ✅ Multi-table JOINs
- ✅ GROUP BY aggregation
- ✅ SQL + Pandas integration
- ✅ Data visualization from queries

**Next:** Assignment 9 - Build your own database from scratch!